# <font color="blue"><b>The Tuning Trilemma: Balancing Performance, Memory, and Cost in Medical Instruction Tuning</b></font>

---

## Problem Statement

Large language models are expensive to adapt. **Full fine-tuning** updates every weight in the model — it works well but needs a lot of GPU memory and compute, which puts it out of reach on modest hardware. Two parameter-efficient methods promise most of the quality for a fraction of the cost:

- **LoRA** freezes the model and trains only small low-rank adapter matrices (well under 1% of the parameters).
- **QLoRA** goes further — it stores the frozen model in **4-bit**, then trains the same LoRA adapters on top, cutting memory again.

In this assignment you will put these three methods on a real task: teaching **Qwen2.5-0.5B-Instruct** to answer medical instructions from the **AlpaCare-MedInstruct-52k** dataset (each example is `instruction (+ optional input) → reference answer`). You will fine-tune the **same model on the same data three ways**, then measure what each approach costs and what it buys.

**The central question:** *given a fixed model, a fixed dataset, and a single free T4 GPU, is it worth updating all ~500M parameters — or does a low-rank slice (LoRA), or a low-rank slice over a 4-bit backbone (QLoRA), get you almost the same answer quality for far less memory?* You will also compare **two LoRA ranks (r = 8 and r = 16)** for both LoRA and QLoRA, to see how adapter capacity trades against cost.

---

## What You Will Do

You will build the pipeline end-to-end, as a hand-written training loop (a torch `Dataset`, a `collate` function, and `DataLoader`s — no `Trainer`), reused across all three methods:

1. **Imports & seed** — set up the environment reproducibly. *(provided)*
2. **Configuration** — model, data sizes, and all hyperparameters in one place. *(provided)*
3. **Helper functions** — LoRA/QLoRA configs, parameter counting, memory utilities. *(provided)*
4. **Load dataset + statistics** — take 300 / 30 / 30 for train / val / test; inspect fields and lengths.
5. **Dataset, collate & loaders** — build the prompt with the chat template, mark answer tokens with an offset-based mask, pad batches, and create the three loaders.
6. **Model load + IFT training** — write the masked-loss + shared training loop, then run **full instruction fine-tuning**.
7. **LoRA** — freeze the backbone and train adapters at **r = 8 and r = 16**.
8. **QLoRA** — repeat the rank sweep with a **4-bit NF4** backbone.
9. **Test evaluation** — greedy-generate on the test set and score every run with **ROUGE-L** and **BERTScore-F1**.
10. **Plots** — per-epoch train vs validation loss for every run.
11. **Error analysis** — inspect the best model's answers vs the references and summarise the failure modes.
12. **Parameter analysis** — compare quality against % trainable parameters, peak memory, and training time.

**How the notebook is organised**
- Cells marked **`# ── PROVIDED ──`** are already written — just run them in order.
- You only need to write the code inside **`## CODE HERE ##`**.
- Every step with a red **`[marks]`** tag is graded: complete the code where indicated and write the **`Your Answer`** reflection.
- Reflections must **cite the actual numbers your run produced** — not generic descriptions.

**Runtime:** Colab **free T4 GPU**. We deliberately use a small data subset and short runs so all methods finish in one session. Select the T4 runtime and run top-to-bottom after a restart. No API key needed.

> **Safety note.** This is an exercise on a 0.5B model with machine-generated references. Its outputs are **not** clinically reliable and must never guide real medical decisions.

---

## Mark Distribution

| Step | Topic | Marks |
|------|-------|-------|
| 1 | Imports & seed | 0 *(provided)* |
| 2 | Configuration | 0 *(provided)* |
| 3 | Helper functions (LoRA / QLoRA) | 0 *(provided)* |
| 4 | Load dataset + statistics | 8 |
| 5 | Dataset, collate & loaders | 14 |
| 6 | Model load + IFT training | 20 |
| 7 | LoRA (r = 8 and r = 16) | 14 |
| 8 | QLoRA (r = 8 and r = 16) | 14 |
| 9 | Test evaluation (ROUGE-L + BERTScore) | 10 |
| 10 | Plots (train vs validation) | 6 |
| 11 | Error analysis | 8 |
| 12 | Parameter analysis | 6 |
| | **Total** | **100** |

Most marks sit where the core learning is: the training loop and full IFT (Step 6), and the data pipeline plus the two PEFT methods (Steps 5, 7, 8). Steps 1–3 are provided scaffolding and carry no marks.

---

> **Learner note:** Use a fresh T4 runtime and execute all cells. Reserve EOS within `MAX_LEN`, ensure at least one supervised answer token, and compare LoRA/QLoRA at matched ranks. Report isolated GPU-memory measurements when possible.

In [ ]:
# Some package installation.
!pip install -q evaluate rouge_score bert_score "bitsandbytes>=0.43"

In [ ]:
# Avoid a torchao/T4 incompatibility that can surface with peft/quantised stacks.
!pip uninstall -y -q torchao

### <font color="blue"><b>Step 1: Imports and Seed Setting</b></font>  *(provided — just run)*

Import the libraries used throughout — PyTorch and `torch.utils.data`, Hugging Face `transformers` / `datasets` / `peft`, and plotting — and fix a single random seed across `random`, NumPy, and PyTorch (`seed_everything`) so every run is reproducible.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
# 1. Imports and Seed Setting
# ---PROVIDED ------------------------------------------------
# ============================================================
import os, gc, math, time, random, warnings, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, get_cosine_schedule_with_warmup,
                          set_seed)
from peft import (LoraConfig, get_peft_model, prepare_model_for_kbit_training,
                  TaskType)

warnings.filterwarnings("ignore")

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

seed_everything()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

### <font color="blue"><b>Step 2: Configuration</b></font>  *(provided — just run)*

All the knobs in one place: the model ID, dataset split sizes (`N_TRAIN` / `N_VAL` / `N_TEST`), the sequence-length budget (`MAX_LEN`), and the training hyperparameters — batch size, gradient accumulation, epochs, the two learning rates (`LR_FULL`, `LR_LORA`), and the LoRA ranks. The rest of the notebook reads from these constants, so change behaviour here rather than editing later cells.

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 2: Configuration
# ---PROVIDED ------------------------------------------------
# ============================================================
MODEL_ID   = "Qwen/Qwen2.5-0.5B-Instruct"

# ---- Data ----
N_TRAIN    = 300
N_VAL      = 30
N_TEST     = 30
MAX_LEN    = 256        # token budget per (prompt + answer) example

# ---- Training ----
BATCH_SIZE = 4          # small: fits a free T4 - if not fit please reduce it
GRAD_ACCUM = 1          # effective batch = BATCH_SIZE * GRAD_ACCUM
EPOCHS     = 3
WARMUP_RATIO  = 0.05
MAX_GRAD_NORM = 1.0

LR_FULL    = 1e-5       # full fine-tuning: small LR (all weights move)
LR_LORA    = 2e-4       # LoRA / QLoRA: frozen backbone -> larger LR is safe

# ---- LoRA / QLoRA ----
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# ---- Evaluation ----
EVAL_N     = N_TEST         # number of held-out test prompts to score

### <font color="blue"><b>Step 3: Helper Functions</b></font>  *(provided — just run)*

Small utilities reused across every method: `count_trainable` (trainable-vs-total parameter counts), `get_lora_config` / `get_bnb_config` (the shared LoRA and 4-bit NF4 quantisation settings), and `gpu_mem_gb` / `free_memory` (peak-memory reporting and VRAM cleanup between runs).

> `# ── PROVIDED ──` This cell is written for you. Run it and move on.

In [ ]:
# ============================================================
#  Step 3: Helper Functions (LoRA / QLoRA)
# ---PROVIDED ------------------------------------------------
# ============================================================

def count_trainable(model):
    """Return (trainable, total, percent-trainable) parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total, 100 * trainable / total


def get_lora_config():
    """Standard LoRA config for Qwen2.5 (attention + MLP projections)."""
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )


def get_bnb_config():
    """4-bit NF4 quantisation config for QLoRA (fp16 compute for T4)."""
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def gpu_mem_gb():
    """Peak GPU memory used since the last reset, in GB."""
    return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0


def free_memory(*objs):
    """Delete big objects (models/trainers) and reclaim GPU memory between runs."""
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

## <font color="blue"><b>Step 4: Load Dataset and Statistics</b></font>  <font color="red">**[8 marks]**</font>

Load **AlpaCare-MedInstruct-52k** and randomly take **300 train / 30 validation / 30 test**. Each row is just `instruction (+ optional input) → output`, where `output` is a reference answer. The dataset stores the literal token `<noinput>` when there is no extra context, so blank that out first.

Then compute a few basic statistics on the training split: the fraction of rows with a non-empty `input`, and the token-length distribution of `instruction + input + output` (mean / median / p95 / max), plus a length histogram.

**Marks:** code **[<font color="red">4 marks</font>]** — correct load, `<noinput>` handling, 300/30/30 split, and the statistics + histogram; reflection **[<font color="red">4 marks</font>]** — see below.

In [ ]:
# ============================================================
#  Step 4: Load Dataset and Statistics
# ============================================================

# load the dataset and convert to pandas DataFrame
DATASET_IDS = ["lavita/AlpaCare-MedInstruct-52k",   # public HF copy of AlpaCare-MedInstruct-52k
               "xz97/MedInstruct"]                  # the authors' original upload (fallback mirror)
raw = None
for ds_id in DATASET_IDS:
    try:
        raw = load_dataset(ds_id, split="train")
        print(f"Loaded {ds_id}: {len(raw):,} rows | columns = {raw.column_names}")
        break
    except Exception as e:
        print(f"[warn] could not load {ds_id}: {type(e).__name__}: {str(e)[:150]}")
if raw is None:
    raise RuntimeError("AlpaCare-MedInstruct-52k could not be downloaded - check the runtime's internet access.")

df = raw.to_pandas()[["instruction", "input", "output"]].fillna("").astype(str)
for col in ["instruction", "input", "output"]:
    df[col] = df[col].str.strip()

# The dataset stores the literal token "<noinput>" when there is no extra context.
is_noinput = df["input"].str.fullmatch(r"(?i)<\s*no\s*input\s*>")
print(f"'<noinput>' placeholders blanked: {int(is_noinput.sum()):,} of {len(df):,} rows ({is_noinput.mean():.1%})")
df.loc[is_noinput, "input"] = ""
n_before = len(df)   # hygiene: every example needs an instruction and a non-empty reference to supervise
df = df[(df["instruction"] != "") & (df["output"] != "")].reset_index(drop=True)
print(f"Dropped {n_before - len(df)} rows with an empty instruction/output -> {len(df):,} usable rows")

# Shuffle once, then slice off the splits by index.
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)      # ONE seeded shuffle of the pool
train_df = df.iloc[:N_TRAIN].reset_index(drop=True)                      # rows [0, 300)
val_df   = df.iloc[N_TRAIN:N_TRAIN + N_VAL].reset_index(drop=True)       # rows [300, 330)
test_df  = df.iloc[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST].reset_index(drop=True)   # rows [330, 360)
assert (len(train_df), len(val_df), len(test_df)) == (N_TRAIN, N_VAL, N_TEST)
print(f"Split sizes -> train {len(train_df)} | val {len(val_df)} | test {len(test_df)}  "
      f"(disjoint index ranges after one shuffle, seed={SEED})")

# ---- Basic statistics on the training split ----
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

frac_with_input = (train_df["input"] != "").mean()

def combined_len(r):
    text = r["instruction"] + " " + r["input"] + " " + r["output"]
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

lens = train_df.apply(combined_len, axis=1)
len_stats = {"mean": lens.mean(), "median": lens.median(), "p95": lens.quantile(0.95), "max": lens.max()}

print("\nFields: instruction = the task/question | input = optional extra context | output = reference answer")
print(f"Rows with a non-empty `input` (train): {frac_with_input:.1%} "
      f"({int((train_df['input'] != '').sum())}/{len(train_df)})")

field_tok = pd.DataFrame({c: train_df[c].map(lambda s: len(tokenizer(s, add_special_tokens=False)["input_ids"]))
                          for c in ["instruction", "input", "output"]})
print("\nPer-field token lengths on the train split:")
print(field_tok.describe(percentiles=[0.5, 0.95]).T[["mean", "50%", "95%", "max"]].round(1).to_string())

print("\nCombined instruction + input + output token length (train):")
for k, v in len_stats.items():
    print(f"  {k:>6}: {v:7.1f}")
over = int((lens > MAX_LEN).sum())
print(f"  rows longer than MAX_LEN={MAX_LEN}: {over}/{len(lens)} ({over / len(lens):.1%})  "
      f"(the chat template adds a fixed overhead on top - measured in Step 5)")
print(f"\n📌 [S4] non-empty input = {frac_with_input:.1%} | mean {len_stats['mean']:.1f} | median {len_stats['median']:.0f} | "
      f"p95 {len_stats['p95']:.0f} | max {len_stats['max']:.0f} | >{MAX_LEN}: {over / len(lens):.1%} | "
      f"<noinput> rows in full set: {is_noinput.mean():.1%}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(lens, bins=30, color="#2a78d6", edgecolor="white", linewidth=1.5)
for x, lab, ls, col in [(len_stats["median"], "median", ":", "#52514e"),
                        (len_stats["p95"], "p95", "--", "#52514e"),
                        (MAX_LEN, "MAX_LEN", "-", "#eb6834")]:
    ax.axvline(x, color=col, linestyle=ls, linewidth=2, label=f"{lab} = {x:.0f}")
ax.set_title("Token length of instruction + input + output (train split, n = 300)")
ax.set_xlabel("Tokens (Qwen2.5 tokenizer)")
ax.set_ylabel("Number of examples")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### ✍️ Step 4 Reflection  <font color="red">[4 marks]</font>

**[Your Answer]** must cover:
- The three fields (`instruction`, `input`, `output`), what each holds, and roughly what fraction of rows use a non-empty `input`. **<font color="red">1 marks</font>**
- The token-length statistics you printed (mean / median / p95 / max) and the `MAX_LEN` you will adopt because of them — justify the choice against the p95. **<font color="red">2 marks</font>**
- That `output` is a *machine-generated* reference (text-davinci-003), and what that implies for treating it as ground truth when scoring models later. **<font color="red">1 marks</font>**

*Cite the actual numbers your run produced.*

### ✅ Your Answer — Step 4

**The three fields.** Every AlpaCare row is an Alpaca-style triple:
- `instruction` is the task or question (e.g. *"List the symptoms of …"*, *"Explain … to a patient"*).
- `input` is optional extra context the instruction refers to, such as a patient vignette, lab values or a passage to summarise. When no context is needed, the dataset stores the literal string `<noinput>`. I blank that to `""` so it never leaks into the prompt. This affected **⟦S4: `<noinput>` rows in full set⟧** of the 52k pool.
- `output` is the reference answer. It is the training target and the yardstick for evaluation.

In the 300-row training split, **⟦S4: non-empty input %⟧** of rows have a non-empty `input`; the rest are instruction-only.

**Token lengths → `MAX_LEN`.** The combined instruction + input + output length (Qwen2.5 tokenizer, train split) is:

| mean | median | p95 | max |
|---|---|---|---|
| **⟦S4 mean⟧** | **⟦S4 median⟧** | **⟦S4 p95⟧** | **⟦S4 max⟧** |

**⟦S4: >256 %⟧** of rows exceed 256 tokens before templating. The chat template adds a fixed **⟦S5: template overhead⟧** tokens (system message, role tags, EOS). After templating, **⟦S5: truncated %⟧** of training examples are clipped at 256.

I keep **`MAX_LEN = 256`**, set against the p95 of **⟦S4 p95⟧**:
- **Memory is the binding constraint.** Full IFT runs in fp32. Weights, gradients and AdamW states for 494M parameters already take about 7.9 GB. The 151,936-way logits cost B·L·V·4 bytes, which is 4·256·151,936·4 ≈ 0.62 GB per fp32 copy, and the log-softmax and its backward keep several copies alive. Stretching the budget to cover the p95 would add memory and time in proportion to its length on a 15 GB T4, for the sake of roughly the longest 5% of answers.
- **Truncation is safe by construction.** The prompt and the start of every answer are always kept, EOS is reserved inside the 256 budget, and only the tail of the longest answers is cut.
- **Longer targets would add little.** Generation is capped at 200 new tokens at test time, so supervising answers much longer than that buys little.

**Machine-generated references.** The `output` texts were written by OpenAI's **text-davinci-003** through a self-instruct pipeline seeded with clinician-written tasks. They were not written or checked answer-by-answer by clinicians. Three consequences:
- They can contain factual errors, outdated guidance or missing caveats. A model that reproduces them well is not thereby clinically correct.
- They share davinci-003's style: long, list-like, often closing with a generic "consult a healthcare professional". ROUGE-L and BERTScore therefore partly reward imitating that style.
- An open medical question has many valid answers, and the reference is only one sample.

I treat the outputs as **silver labels**. They are a common yardstick for *comparing* methods, not ground truth for medical accuracy.

## <font color="blue"><b>Step 5: Dataset, Collate, and DataLoaders</b></font>  <font color="red">**[14 marks]**</font>

Build the input pipeline as a hand-written `torch.utils.data.Dataset` (like the DPO demo), with tokenisation deferred to a `collate` function:

- **`SFTDataset`** — holds `(prompt, answer)` text pairs. The prompt is rendered with the tokenizer's **chat template** (instruction → *user* turn, primed to answer).
- **`tokenize_example`** — tokenises the full `prompt + answer` **once** and uses the **offset mapping** to mark exactly where the answer begins, producing a `comp_mask` (1 = answer token, 0 = prompt token).
- **`collate`** — pads a batch to its longest sequence and returns `input_ids`, `attn`, and `comp_mask`.
- **Three DataLoaders** — `train_loader` (shuffled), `val_loader`, `test_loader`.

Finally, print the **first full training batch** to confirm the shapes and the answer/prompt masking.

**Marks:** code <font color="red">**[10 marks]**</font> — `SFTDataset` + chat-template prompt, offset-based `comp_mask`, correct padded `collate`, three loaders, and the batch print; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 5: Dataset, Collate, and DataLoaders
# ============================================================

SYSTEM_MSG = "You are a helpful medical assistant. Answer accurately and safely."

def build_prompt(instruction, input_text):
    user_msg = instruction.strip()
    if input_text and input_text.strip():                 # optional context goes under the instruction
        user_msg += "\n\nInput:\n" + input_text.strip()
    msgs = [{"role": "system", "content": SYSTEM_MSG},
            {"role": "user",   "content": user_msg}]      # add_generation_prompt=True primes the assistant turn
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


class SFTDataset(Dataset):
    """Holds (prompt_text, answer_text) pairs; tokenisation is deferred to collate()."""
    def __init__(self, frame):
        self.rows = [{"prompt": build_prompt(r["instruction"], r["input"]),
                      "answer": r["output"].strip()}
                     for _, r in frame.iterrows()]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        return self.rows[i]


# Tokens that close the user turn and open the assistant turn ("<|im_end|>\n<|im_start|>assistant\n").
_probe = build_prompt("X", "")
GEN_SUFFIX_LEN = len(tokenizer(_probe[_probe.rindex("X") + 1:], add_special_tokens=False)["input_ids"])
MIN_ANSWER_TOKENS = 16   # an over-long prompt can never crowd out the answer completely

def tokenize_example(prompt, answer):
    """
    Tokenise the full (prompt + answer) once and use the offset mapping to mark
    exactly where the answer begins. comp_mask = 1 on answer tokens, 0 on prompt.
    Tokenising once avoids token-merge drift at the boundary.
    """
    enc = tokenizer(prompt + answer, add_special_tokens=False, return_offsets_mapping=True)
    ids = list(enc["input_ids"])
    ans_start = len(prompt)                                   # first character of the answer
    mask = [1 if end > ans_start else 0 for (_, end) in enc["offset_mapping"]]   # token covers answer text

    budget = MAX_LEN - 1                                      # reserve the final slot for EOS
    n_prompt = mask.index(1) if 1 in mask else len(mask)
    if n_prompt > budget - MIN_ANSWER_TOKENS:                 # rare: trim the prompt's middle, keep the
        keep_head = max(budget - MIN_ANSWER_TOKENS - GEN_SUFFIX_LEN, 0)   # header + the "assistant\n" cue
        ids  = ids[:keep_head]  + ids[n_prompt - GEN_SUFFIX_LEN:]
        mask = mask[:keep_head] + mask[n_prompt - GEN_SUFFIX_LEN:]
    ids, mask = ids[:budget], mask[:budget]                   # long answers lose their tail, never their start
    ids.append(tokenizer.eos_token_id)                        # EOS (<|im_end|>) always fits inside MAX_LEN ...
    mask.append(1)                                            # ... and is supervised, so the model learns to stop
    return ids, mask


def collate(batch):
    """Pad a batch to its longest sequence. Returns input_ids, attn, comp_mask."""
    pairs = [tokenize_example(ex["prompt"], ex["answer"]) for ex in batch]
    longest = max(len(ids) for ids, _ in pairs)
    input_ids = torch.full((len(pairs), longest), tokenizer.pad_token_id, dtype=torch.long)
    attn      = torch.zeros((len(pairs), longest), dtype=torch.long)
    comp_mask = torch.zeros((len(pairs), longest), dtype=torch.long)
    for i, (ids, m) in enumerate(pairs):                      # right padding: real tokens first
        n = len(ids)
        input_ids[i, :n] = torch.tensor(ids, dtype=torch.long)
        attn[i, :n]      = 1                                  # pads stay 0 -> never attended to
        comp_mask[i, :n] = torch.tensor(m, dtype=torch.long)  # pads stay 0 -> never supervised
    return input_ids, attn, comp_mask


# ---- Three DataLoaders ----
train_ds, val_ds, test_ds = SFTDataset(train_df), SFTDataset(val_df), SFTDataset(test_df)
loader_gen = torch.Generator().manual_seed(SEED)              # train_model() re-seeds it -> same batch order every run
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate, generator=loader_gen)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# ---- Print the first FULL batch of the train loader ----
input_ids, attn, comp_mask = next(iter(train_loader))
print(f"\ninput_ids {tuple(input_ids.shape)} {input_ids.dtype} | attn {tuple(attn.shape)} | comp_mask {tuple(comp_mask.shape)}")
print(f"pad id = {tokenizer.pad_token_id} ({tokenizer.pad_token!r}) | eos id = {tokenizer.eos_token_id} ({tokenizer.eos_token!r})")
torch.set_printoptions(threshold=1_000_000, linewidth=170)
print("\ninput_ids =\n", input_ids)
print("\nattn =\n", attn)
print("\ncomp_mask =\n", comp_mask)
torch.set_printoptions(profile="default")

real    = attn.sum(1)
sup     = comp_mask.sum(1)
prompt_ = real - sup
pad     = input_ids.shape[1] - real
print("\nrow | real tokens | prompt (masked, 0) | answer+EOS (supervised, 1) | padding (masked, 0)")
for i in range(input_ids.shape[0]):
    print(f" {i}  | {int(real[i]):11d} | {int(prompt_[i]):18d} | {int(sup[i]):26d} | {int(pad[i]):19d}")
tot = input_ids.numel()
print(f"batch | supervised = {int(sup.sum())} ({sup.sum() / tot:.1%}) | masked prompt = {int(prompt_.sum())} "
      f"({prompt_.sum() / tot:.1%}) | masked padding = {int(pad.sum())} ({pad.sum() / tot:.1%}) | total = {tot}")
assert all(input_ids[i, real[i] - 1].item() == tokenizer.eos_token_id for i in range(len(real))), "EOS missing"
assert (comp_mask * (1 - attn)).sum() == 0, "a padding position is supervised"

# Decode row 0 split by the mask, to eyeball the boundary.
row_ids, row_m = input_ids[0, :real[0]].tolist(), comp_mask[0, :real[0]].tolist()
print("\n--- row 0, PROMPT part (comp_mask = 0) ---")
print(tokenizer.decode([t for t, m in zip(row_ids, row_m) if m == 0]))
print("--- row 0, ANSWER part (comp_mask = 1) ---")
print(tokenizer.decode([t for t, m in zip(row_ids, row_m) if m == 1])[:600])

# Whole-train-split check of the masking/truncation rules.
full_lens, sup_counts, prompt_lens = [], [], []
for ex in train_ds.rows:
    ids_, m_ = tokenize_example(ex["prompt"], ex["answer"])
    full_lens.append(len(tokenizer(ex["prompt"] + ex["answer"], add_special_tokens=False)["input_ids"]) + 1)
    sup_counts.append(sum(m_))
    prompt_lens.append(len(m_) - sum(m_))
full_lens = np.array(full_lens)
overhead = len(tokenizer(build_prompt("", ""), add_special_tokens=False)["input_ids"]) + 1
print(f"\nChat-template overhead (system + role tags + EOS): {overhead} tokens per example")
print(f"Templated prompt+answer+EOS length (train): mean {full_lens.mean():.1f} | median {np.median(full_lens):.0f} | "
      f"p95 {np.percentile(full_lens, 95):.0f} | max {full_lens.max()}")
print(f"Examples truncated by MAX_LEN={MAX_LEN}: {(full_lens > MAX_LEN).sum()}/{len(full_lens)} "
      f"({(full_lens > MAX_LEN).mean():.1%}) | min supervised tokens/example = {min(sup_counts)} | "
      f"mean prompt tokens = {np.mean(prompt_lens):.1f} | mean supervised tokens = {np.mean(sup_counts):.1f}")
print(f"\n📌 [S5] batch {tuple(input_ids.shape)} | supervised {int(sup.sum())} | masked prompt {int(prompt_.sum())} | "
      f"padding {int(pad.sum())} | template overhead {overhead} | truncated {(full_lens > MAX_LEN).mean():.1%}")

### ✍️ Step 5 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- Why the prompt uses the tokenizer's **chat template** for an `-Instruct` model (special tokens must match what the model saw in pre-training). <font color="red">**[1]**</font>
- What `comp_mask` encodes and why we compute the loss on the **answer tokens only** (completion-only). <font color="red">**[1]**</font>
- Why tokenising `prompt + answer` **once with offsets** is more correct than tokenising them separately and concatenating (avoids token-merge drift at the boundary). <font color="red">**[1]**</font>
- From the printed batch: the supervised-vs-masked token counts, and why padding fills `input_ids` with the pad token but `comp_mask` with 0. <font color="red">**[1]**</font>

*Cite the actual numbers your run produced.*

### ✅ Your Answer — Step 5

**Why the chat template.** Qwen2.5-0.5B-*Instruct* was post-trained on ChatML. Every turn is `<|im_start|>role\n … <|im_end|>\n`, and `<|im_start|>assistant\n` cues the reply. These are dedicated special tokens whose embeddings were learned in exactly this arrangement. `apply_chat_template(..., add_generation_prompt=True)` reproduces that layout exactly: system turn, then user turn, then an open assistant turn. This gives three benefits:
- Fine-tuning only has to change *what* the model says, not teach it a new format.
- `generate_answers` builds prompts with the same `build_prompt`, so training and test prompts are identical.
- The supervised answer ends with `<|im_end|>`, the token `generate()` stops on.

A home-made `### Instruction:` format would be out of distribution for an Instruct model, spend the 300 examples on learning formatting, and leave the stop signal undefined.

**What `comp_mask` encodes.** `comp_mask[i, t] = 1` exactly when token *t* belongs to the reference answer or is the final `<|im_end|>` (EOS). It is 0 for the system message, the user turn, the role tags and all padding. The loss averages next-token NLL only over positions whose **target** has `comp_mask = 1` (completion-only).

At inference the prompt is given, never generated, so there is nothing useful to learn from predicting it. On average **⟦S5: mean prompt tokens⟧** prompt tokens per example would otherwise dilute the gradient, and the system message is identical across all 300 rows, so most of those tokens are trivially predictable. That would make the loss look artificially low and could teach the model to write user turns. Supervising EOS teaches the model *when to stop*.

**Why tokenise once with offsets.** BPE merges depend on neighbouring characters, so `tok(prompt) + tok(answer)` is not guaranteed to equal `tok(prompt + answer)`. At the seam, characters can merge into a different token ("token-merge drift"). The training ids would then differ from what the model sees at inference, and the mask boundary could be off by one.

I tokenise the full string once and request `return_offsets_mapping`. Each token then carries its character span, and a token is marked as answer if its span ends after `len(prompt)`. The boundary is exact even if a token straddles it. In the printed decode of row 0, the masked part ends exactly at `<|im_start|>assistant\n` and the supervised part is the answer plus `<|im_end|>`.

**The printed batch.** Shape **⟦S5: batch shape⟧**:
- **⟦S5: supervised⟧** supervised answer and EOS tokens
- **⟦S5: masked prompt⟧** masked prompt tokens
- **⟦S5: padding⟧** padding positions

Padding fills `input_ids` with the pad id `<|endoftext|>` (151643) because the tensor must be rectangular and every slot needs a valid vocabulary id for the embedding lookup. That id is deliberately distinct from EOS `<|im_end|>` (151645), and `attn = 0` there, so real tokens never attend to it.

`comp_mask` is 0 on padding because padding is not part of any answer. Supervising it would train the model to emit filler after EOS, and the loss would depend on how much padding a batch happens to contain. The code asserts that no padded position is supervised.

## <font color="blue"><b>Step 6: Model Loading and Full Instruction Fine-Tuning (IFT)</b></font>  <font color="red">**[20 marks]**</font>

This is the core of the notebook. First define the training machinery that all three methods reuse, then run full fine-tuning:

- **`batch_loss`** — the causal-LM next-token loss, but computed over **answer tokens only** (using `comp_mask`), so the model is never scored on re-typing the prompt.
- **`eval_loss`** — averages `batch_loss` over a whole loader (no gradients), used for validation.
- **`train_model`** — the shared manual training loop: gradient accumulation, warmup → cosine schedule, gradient clipping, per-epoch train/val history, and early stopping on validation loss. Returns the best model plus its history, training time, and peak memory.

Then load Qwen2.5-0.5B in **fp32** (pure fp16 full fine-tuning overflows on this model), confirm ~100% of parameters are trainable, run **full IFT** with the small learning rate `LR_FULL`, and print the per-epoch train/val losses.

**Marks:** code <font color="red">**[14 marks]**</font> — correct answer-only `batch_loss`, the `train_model` loop (accumulation, schedule, clipping, early stopping, history), fp32 model load, and the IFT run with per-epoch report; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 6: Load Model and Full Instruction Fine-Tuning (IFT)
# ============================================================

# ---- Loss over ANSWER tokens only (comp_mask == 1) ----
def batch_loss(model, input_ids, attn, comp_mask):
    """Mean next-token NLL over answer tokens for ONE batch."""
    input_ids, attn, comp_mask = input_ids.to(DEVICE), attn.to(DEVICE), comp_mask.to(DEVICE)
    logits = model(input_ids=input_ids, attention_mask=attn, use_cache=False).logits
    # causal shift: the logits at position t predict the token at position t+1
    logp = F.log_softmax(logits[:, :-1, :].float(), dim=-1)            # fp32 log-probs (stable for fp16 compute too)
    tok  = logp.gather(-1, input_ids[:, 1:].unsqueeze(-1)).squeeze(-1)  # log p(true next token), shape [B, T-1]
    mask = comp_mask[:, 1:].to(tok.dtype)                               # 1 where the TARGET is an answer/EOS token
    return -(tok * mask).sum() / mask.sum().clamp(min=1.0)


@torch.no_grad()
def eval_loss(model, loader):
    """Average batch_loss over an entire loader (no gradients)."""
    was_training = model.training
    model.eval()                                   # no dropout during evaluation
    tot, n = 0.0, 0
    for input_ids, attn, comp_mask in loader:
        tot += batch_loss(model, input_ids, attn, comp_mask).item()
        n += 1
    if was_training:
        model.train()
    return tot / max(n, 1)


# ---- Shared training loop (reused for IFT, LoRA, QLoRA) ----
def train_model(model, lr, tag):
    """
    Manual training loop: gradient accumulation, warmup->cosine schedule,
    gradient clipping, and early stopping on validation loss.
    Returns (best_model, hist, train_time_s, peak_mem_gb).
    hist keys: 'train_loss' & 'val_loss' (per epoch), 'step_loss' (per step).
    """
    seed_everything()
    free_memory()

    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    trainable_keys = {n for n, p in model.named_parameters() if p.requires_grad}   # only these change
    total_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps)

    hist = {"train_loss": [], "val_loss": [], "step_loss": []}
    best_val, best_epoch, best_state, no_improve, PATIENCE = float("inf"), 0, None, 0, 2

    t0 = time.time()

    params = [p for p in model.parameters() if p.requires_grad]
    if torch.cuda.is_available():
        print(f"[{tag}] GPU memory resident at start = {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"[{tag}] lr={lr:g} | effective batch = {BATCH_SIZE}x{GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} | "
          f"optimizer steps = {total_steps} (warmup {int(WARMUP_RATIO * total_steps)})")
    train_loader.generator.manual_seed(SEED)       # identical shuffling for every run -> fair comparison
    step = 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        ep_loss, n_batches, window = 0.0, 0, []
        for it, (input_ids, attn, comp_mask) in enumerate(train_loader, 1):
            loss = batch_loss(model, input_ids, attn, comp_mask)
            (loss / GRAD_ACCUM).backward()                         # gradient accumulation (scaled)
            ep_loss += loss.item(); n_batches += 1; window.append(loss.item())
            if it % GRAD_ACCUM == 0 or it == len(train_loader):    # one optimizer step per GRAD_ACCUM micro-batches
                torch.nn.utils.clip_grad_norm_(params, MAX_GRAD_NORM)   # gradient clipping (global L2 norm <= 1)
                optimizer.step()
                scheduler.step()                                    # linear warmup, then cosine decay
                optimizer.zero_grad(set_to_none=True)
                step += 1
                hist["step_loss"].append(sum(window) / len(window)); window = []
        train_loss = ep_loss / max(n_batches, 1)
        val_loss = eval_loss(model, val_loader)
        hist["train_loss"].append(train_loss)
        hist["val_loss"].append(val_loss)
        print(f"[{tag}] epoch {epoch}/{EPOCHS} | train {train_loss:.4f} | val {val_loss:.4f} | "
              f"lr {scheduler.get_last_lr()[0]:.2e} | step {step}/{total_steps} | {time.time() - t0:.0f}s")

        if val_loss < best_val:                                     # early-stopping bookkeeping
            best_val, best_epoch, no_improve = val_loss, epoch, 0
            # snapshot ONLY the trainable tensors, on CPU (4-bit quant-state buffers are never copied)
            best_state = {k: v.detach().to("cpu", copy=True)
                          for k, v in model.state_dict().items() if k in trainable_keys}
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"[{tag}] early stopping: no val improvement for {PATIENCE} epochs")
                break

    if best_state is not None:                                      # restore the best-validation weights
        res = model.load_state_dict(best_state, strict=False)       # strict=False: trainable params only
        assert not res.unexpected_keys, res.unexpected_keys
    model.eval()

    train_time, peak_mem = time.time() - t0, gpu_mem_gb()
    print(f"[{tag}] best epoch = {best_epoch} | time = {train_time:.1f}s | peak mem = {peak_mem:.2f} GB")
    return model, hist, train_time, peak_mem


# ---- Load model for Full IFT ----
# fp32 (not fp16): pure-fp16 full fine-tuning overflows on Qwen2.5 and gives a
# garbage loss in the hundreds. fp32 is fine for a 0.5B model on a T4.
seed_everything()
free_memory()

import transformers
from packaging import version
# transformers >= 4.56 renamed `torch_dtype` to `dtype`; pick whichever this runtime understands
DTYPE_KW = "dtype" if version.parse(transformers.__version__) >= version.parse("4.56.0") else "torch_dtype"

def park(model, tag):
    """After training, move a full-precision model to CPU RAM (so the next run's GPU peak is isolated) when
    at least 3 GB of RAM would remain; otherwise keep it on the GPU - its memory is then subtracted as 'resident'."""
    need = sum(p.numel() * p.element_size() for p in model.parameters())
    try:
        import psutil
        room = psutil.virtual_memory().available > need + 3e9
    except ImportError:
        room = False
    if DEVICE == "cuda" and room:
        model.to("cpu")
    free_memory()
    print(f"[{tag}] trained model kept on {'CPU RAM' if room or DEVICE != 'cuda' else 'GPU (not enough RAM to park)'}")

full_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: torch.float32}).to(DEVICE)
full_model.config.use_cache = False                                  # no KV cache while training
full_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
tr_full, tot_full, pct_full = count_trainable(full_model)
print(f"Full IFT | dtype = {next(full_model.parameters()).dtype} | trainable = {tr_full:,} / {tot_full:,} "
      f"({pct_full:.2f}%) | gradient checkpointing = {full_model.is_gradient_checkpointing}")

full_model, full_hist, full_time, full_mem = train_model(full_model, LR_FULL, "full_ift")

park(full_model, "full_ift")      # isolate the next runs' GPU-memory readings
print(f"\n📌 [S6] trainable {tr_full:,} ({pct_full:.2f}%) | time {full_time:.1f}s | peak mem {full_mem:.2f} GB | "
      f"best val {min(full_hist['val_loss']):.4f} (epoch {int(np.argmin(full_hist['val_loss'])) + 1})")

# ---- Per-epoch loss history ----
print("\nEpoch |  Train Loss | Val Loss")
print("-" * 34)
for ep, (tl, vl) in enumerate(zip(full_hist["train_loss"], full_hist["val_loss"]), 1):
    print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")

### ✍️ Step 6 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- How `batch_loss` supervises **only** the answer tokens (the `comp_mask` shift + masked mean), and how completion-only loss differs from training on the whole sequence. <font color="red">**[2 marks]**</font>
- Where warmup, cosine decay, gradient clipping, and gradient accumulation appear in `train_model`, and your **effective batch size** (`BATCH_SIZE × GRAD_ACCUM`). <font color="red">**[2 marks]**</font>
- Why full IFT is loaded in **fp32** (pure fp16 full fine-tuning overflows on Qwen2.5) and why its learning rate `LR_FULL = 1e-5` is deliberately small. <font color="red">**[1 marks]**</font>
- The per-epoch train/val trend — is the model improving or starting to overfit? <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

### ✅ Your Answer — Step 6

**How `batch_loss` supervises only the answer.**
- **Shift.** A causal LM's logits at position *t* are a distribution over token *t+1*. `batch_loss` therefore aligns `logits[:, :-1]` with the targets `input_ids[:, 1:]`.
- **Per-token log-probability.** It applies `log_softmax` in fp32 and uses `gather` to read the log-probability of each true next token. The result `tok` has shape [B, T−1].
- **Shifted mask.** The mask is shifted the same way (`comp_mask[:, 1:]`), so position *t* counts exactly when its *target* is an answer or EOS token. The first answer token is predicted from the last prompt token (`assistant\n`), and no prompt or pad target ever contributes.
- **Masked mean.** The loss is `−Σ(tok·mask) / Σmask`, the mean NLL per supervised token across the batch. It matches HF's built-in loss with `labels = −100` on masked positions.

Whole-sequence training would also average in the prompt tokens. Those are easy and largely repeated (the same system message in every row), so the loss would look lower, the gradient would be spent on copying the prompt, and the model would partly learn to produce system and user text.

**Where the optimisation pieces sit in `train_model`.**
- **Gradient accumulation:** each micro-batch loss is divided by `GRAD_ACCUM` before `.backward()`. The optimiser steps only when `it % GRAD_ACCUM == 0` or on the last batch.
- **Gradient clipping:** `clip_grad_norm_(params, 1.0)` runs right before every `optimizer.step()`, capping the global gradient L2 norm so one unusual batch cannot cause a huge update.
- **Warmup → cosine:** `get_cosine_schedule_with_warmup(opt, int(0.05·225) = 11, 225)`, with `scheduler.step()` after each optimiser step. The LR ramps linearly from 0 to peak over 11 steps, while AdamW's moment estimates are still noisy, then follows a half-cosine down to 0 at step 225.
- **Effective batch:** `BATCH_SIZE × GRAD_ACCUM = 4 × 1 = 4` sequences per update, which gives 75 updates per epoch and 225 in total.
- **Also:** AdamW over trainable parameters only, per-epoch train/val history, and early stopping with patience 2. The best epoch's *trainable* tensors are snapshotted to CPU and restored with `load_state_dict(strict=False)`.

**Why fp32 and a small LR.** fp16 tops out around 65,504 and has an 11-bit significand. In *pure* fp16 full fine-tuning (no fp32 master weights, no loss scaling) two things break on Qwen2.5:
- Some activations and logits overflow to inf/NaN, which shows up as the "loss in the hundreds".
- Updates of size lr × grad ≈ 1e-5 fall below fp16's resolution relative to the weights, so many of them round to zero.

fp32 avoids both problems, and a 0.5B model can afford it: 1.98 GB of weights + 1.98 GB of gradients + 3.95 GB of AdamW moments ≈ 7.9 GB of static memory, under the T4's 15 GB. The measured peak was **⟦S6: peak mem⟧ GB**.

`LR_FULL = 1e-5` is deliberately small because all **494,032,768** weights move. The model is already instruction-tuned and there are only 300 examples. A large LR would overwrite general and instruction-following knowledge (catastrophic forgetting) and overfit within an epoch. A small LR gives a gentle nudge toward the AlpaCare answer style.

**Trend.**

| | Epoch 1 | Epoch 2 | Epoch 3 |
|---|---|---|---|
| Train loss | **⟦e1⟧** | **⟦e2⟧** | **⟦e3⟧** |
| Validation loss | **⟦e1⟧** | **⟦e2⟧** | **⟦e3⟧** |

The best validation loss came at epoch **⟦best epoch⟧**. Validation loss **⟦pick one: keeps falling every epoch / bottoms out at epoch X and then rises⟧**, and the val − train gap at the last epoch is **⟦S10: val_minus_train_last⟧**. So full IFT is **⟦pick one: still improving and not yet overfitting / starting to overfit⟧**.

Note that the epoch-1 *train* loss is averaged over an epoch in which the weights were still changing, so it can sit above the epoch-1 validation loss without anything being wrong.

## <font color="blue"><b>Step 7: LoRA Fine-Tuning — r = 8 and r = 16</b></font>  <font color="red">**[14 marks]**</font>

Now freeze the backbone and train only small low-rank adapter matrices. For a frozen weight `W`, LoRA learns `A` and `B` and adds `(α/r)·B·A` as a correction — so only a tiny fraction of the parameters are trained.

Run LoRA at **two ranks, r = 8 and r = 16**, holding the `alpha/rank` ratio fixed (`alpha = 2·r`) so you isolate the effect of *rank* alone. Reuse the same `train_model` loop, store each run in `lora_runs[rank]`, and report the trainable-parameter percentage and per-epoch losses for both.

*(The `torchao` uninstall here avoids a T4 incompatibility that can otherwise break the peft/quantised path in the next step.)*

**Marks:** code <font color="red">**[8 marks]**</font> — correct `LoraConfig` (rank, alpha, dropout, target modules), the two-rank sweep reusing `train_model`, and the trainable-% + per-epoch report for each rank; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
LORA_RANKS = [8, 16]   # compare two LoRA ranks
# ============================================================
#  Step 7: LoRA Fine-Tuning (rank sweep: r = 8 and r = 16)
# ============================================================

def build_lora_model(rank):
    """Fresh fp32 backbone + LoRA adapters at the given rank."""
    seed_everything()
    free_memory()
    base = AutoModelForCausalLM.from_pretrained(MODEL_ID, **{DTYPE_KW: torch.float32}).to(DEVICE)
    base.config.use_cache = False
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})  # same as IFT/QLoRA
    base.enable_input_require_grads()
    cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=2 * rank,                 # alpha/r = 2 at every rank -> only the rank changes
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",       # attention projections
                        "gate_proj", "up_proj", "down_proj"],         # MLP projections
    )
    seed_everything()                        # identical adapter init for LoRA and QLoRA at the same rank
    return get_peft_model(base, cfg)         # freezes the backbone, injects trainable A/B matrices


# ---- Run LoRA at each rank ----
lora_runs = {}   # rank -> dict(model, hist, time, mem, pct)

for rank in LORA_RANKS:
    foreign_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0   # other models on GPU
    model = build_lora_model(rank)
    print(f"\n================  LoRA  r = {rank}  (alpha = {2 * rank})  ================")
    model.print_trainable_parameters()
    trainable, total = model.get_nb_trainable_parameters()
    pct = 100 * trainable / total

    model, hist, run_time, run_mem = train_model(model, LR_LORA, f"lora_r{rank}")
    run_mem -= foreign_gb                    # isolated peak: exclude memory held by any other resident model
    park(model, f"lora_r{rank}")             # isolate the next run's peak-memory reading
    lora_runs[rank] = dict(model=model, hist=hist, time=run_time, mem=run_mem, pct=pct,
                           trainable=trainable, total=total)
    print(f"📌 [S7 r={rank}] trainable {trainable:,} / {total:,} = {pct:.4f}% | time {run_time:.1f}s | "
          f"peak mem {run_mem:.2f} GB | best val {min(hist['val_loss']):.4f} "
          f"(epoch {int(np.argmin(hist['val_loss'])) + 1})")

    print("\nEpoch |  Train Loss | Val Loss")
    print("-" * 34)
    for ep, (tl, vl) in enumerate(zip(hist["train_loss"], hist["val_loss"]), 1):
        print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")
    print()

    free_memory()   # clear before the next rank

d_val = min(lora_runs[16]["hist"]["val_loss"]) - min(lora_runs[8]["hist"]["val_loss"])
print(f"📌 [S7] best val r16 - r8 = {d_val:+.4f} | params x{lora_runs[16]['trainable'] / lora_runs[8]['trainable']:.1f} | "
      f"vs full IFT: r8 trains {tr_full / lora_runs[8]['trainable']:.0f}x fewer, r16 {tr_full / lora_runs[16]['trainable']:.0f}x fewer")

### ✍️ Step 7 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- The role of the **rank `r`** and **alpha `α`** (the `α/r` scaling), and why `B` is zero-initialised (the adapter starts as identity, so training is stable from step 0). <font color="red">**[2 marks]**</font>
- The **% trainable parameters** at r = 8 vs r = 16 (from `print_trainable_parameters`), and how both compare to full IFT's ~100%. <font color="red">**[2 marks]**</font>
- The effect of doubling the rank on validation loss — did r = 16 actually help, or just cost more parameters for the same quality? Cite both runs. <font color="red">**[1 marks]**</font>
- Why LoRA can safely use a much larger learning rate (`LR_LORA = 2e-4`) than full IFT. <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

### ✅ Your Answer — Step 7

**Rank, alpha and zero-initialised B.** LoRA keeps `W ∈ ℝ^{d×k}` frozen and learns a correction `ΔW = (α/r)·B·A`, with `A ∈ ℝ^{r×k}` and `B ∈ ℝ^{d×r}`.
- **Rank r** is the width of that bottleneck. It caps the rank of ΔW, i.e. how many independent directions the adaptation can use, and sets the cost: `r·(d+k)` parameters per adapted matrix.
- **α** is a scale. The multiplier α/r keeps the size of the update comparable when r changes. I set α = 2r (16 for r = 8, 32 for r = 16), so both runs use α/r = 2 and **only capacity differs, not step size**.
- **B is zero-initialised** while A is random (Kaiming). So `B·A = 0` and the adapted layer is exactly the pre-trained layer at step 0: the adapter starts as an identity correction. The starting loss is the base model's, and no random perturbation hits a good model, so training is stable from step 0. Learning still starts immediately, because `∂L/∂B = (∂L/∂h)(A x)ᵀ ≠ 0` when A ≠ 0. If A were zero as well, the gradient would vanish.

**% trainable vs full IFT.** The adapters sit on 7 projections × 24 layers, which works out to `r·(1792 + 1024 + 1024 + 1792 + 3·5760) = 22,912·r` parameters per layer:

| | Trainable | Total | % trainable | Vs full IFT |
|---|---|---|---|---|
| LoRA r = 8 | **4,399,104** | 498,431,872 | **0.8826%** | 112× fewer |
| LoRA r = 16 | **8,798,208** | 502,830,976 | **1.7497%** | 56× fewer |
| Full IFT | 494,032,768 | 494,032,768 | 100% | — |

The percentages come from `print_trainable_parameters()`. The optimiser state shrinks in proportion: AdamW keeps two fp32 moments per trainable parameter, about 0.07 GB (r = 8) or 0.14 GB (r = 16) for the adapters, versus 3.95 GB for full IFT.

**Effect of doubling the rank.**

| | Best val loss | Epoch |
|---|---|---|
| r = 8 | **⟦S7 r=8 best val⟧** | **⟦⟧** |
| r = 16 | **⟦S7 r=16 best val⟧** | **⟦⟧** |

The difference is **⟦S7: best val r16 − r8⟧**. Test ROUGE-L is **⟦S9 lora_r8⟧** for r = 8 vs **⟦S9 lora_r16⟧** for r = 16. **⟦pick one: r = 16 gave a small but consistent gain, worth its 2× adapter parameters / the gap is within noise for 30 validation prompts, so r = 16 mostly cost twice the parameters for the same quality⟧**. With 300 examples and 225 steps the task is low-data, and a rank-8 subspace is usually already enough to capture the shift in answer style.

**Why LoRA tolerates `LR_LORA = 2e-4` (20× `LR_FULL`).**
- **The backbone is frozen**, so pre-trained knowledge cannot be overwritten. Only a small low-rank correction is learned.
- **ΔW starts at exactly 0** (B = 0) and grows gradually, and its size is tempered by α/r. Large steps on A and B still produce a small, controlled change in W early on.
- **The low-rank parameterisation is itself a strong regulariser.** It gives far fewer ways to overfit than 494M free weights.
- **At 1e-5 the adapters would barely leave zero** in 225 steps. A larger LR is *needed* for them to learn anything.

## <font color="blue"><b>Step 8: QLoRA Fine-Tuning — r = 8 and r = 16</b></font>  <font color="red">**[14 marks]**</font>

LoRA froze the backbone but still kept it in full precision, which dominates GPU memory. **QLoRA** stores that frozen backbone in **4-bit NF4** instead, then trains the *same* LoRA adapters on top — the only thing that changes from Step 7 is the backbone's precision, making this a clean controlled comparison.

Set the 4-bit recipe with `BitsAndBytesConfig` (NF4, double quantisation, fp16 compute for the T4), call `prepare_model_for_kbit_training` before attaching the adapters, and run the **same two-rank sweep (r = 8 and r = 16)**. Store each run in `qlora_runs[rank]` and report the trainable-parameter percentage and per-epoch losses.

**Marks:** code <font color="red">**[8 marks]**</font> — correct `BitsAndBytesConfig`, `prepare_model_for_kbit_training`, the same `LoraConfig` as Step 7, the two-rank sweep reusing `train_model`, and the trainable-% + per-epoch report; reflection <font color="red">**[6 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 8: QLoRA Fine-Tuning (4-bit NF4 backbone; rank sweep: r = 8 and r = 16)
# ============================================================

def build_qlora_model(rank):
    """Fresh 4-bit NF4 backbone + LoRA adapters at the given rank."""
    seed_everything()
    free_memory()

    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,                       # frozen linear weights stored in 4 bits
        bnb_4bit_quant_type="nf4",               # NormalFloat-4 levels, matched to ~Gaussian weights
        bnb_4bit_use_double_quant=True,          # quantise the per-block scales too
        bnb_4bit_compute_dtype=torch.float16,    # de-quantise to fp16 for the matmuls (T4: no bf16)
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_cfg,
        device_map={"": 0} if torch.cuda.is_available() else {"": "cpu"},
        **{DTYPE_KW: torch.float16},
    )
    base.config.use_cache = False
    # BEFORE attaching LoRA: freeze base, up-cast norms/embeddings to fp32, enable gradient checkpointing
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True,
                                           gradient_checkpointing_kwargs={"use_reentrant": False})
    cfg = LoraConfig(                            # exactly the Step-7 adapter recipe
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=2 * rank,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    seed_everything()                            # same adapter initialisation as LoRA at this rank
    return get_peft_model(base, cfg)


# ---- Run QLoRA at each rank ----
qlora_runs = {}   # rank -> dict(model, hist, time, mem, pct)

for rank in LORA_RANKS:

    foreign_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0   # e.g. QLoRA r=8 kept on GPU
    model = build_qlora_model(rank)
    print(f"\n================  QLoRA  r = {rank}  (alpha = {2 * rank}, 4-bit NF4 backbone)  ================")
    model.print_trainable_parameters()           # peft counts each packed 4-bit element as 2 weights
    trainable, total = model.get_nb_trainable_parameters()
    pct = 100 * trainable / total
    raw_tr, raw_tot, raw_pct = count_trainable(model)
    print(f"count_trainable(): {raw_tr:,} / {raw_tot:,} = {raw_pct:.4f}%  <- numel() of packed uint8 storage "
          f"(2 x 4-bit weights per element) shrinks the denominator; the trainable count itself is unchanged")
    lc = lora_runs[rank]["model"].peft_config["default"]
    qc = model.peft_config["default"]
    same_cfg = all(getattr(lc, k) == getattr(qc, k) for k in ["r", "lora_alpha", "lora_dropout", "bias", "target_modules"])
    print(f"Same LoraConfig as LoRA r={rank}: {same_cfg} | same trainable count: {trainable == lora_runs[rank]['trainable']}")
    assert same_cfg and trainable == lora_runs[rank]["trainable"]

    model, hist, run_time, run_mem = train_model(model, LR_LORA, f"qlora_r{rank}")
    run_mem -= foreign_gb                        # isolated peak (4-bit models stay on the GPU for evaluation)
    qlora_runs[rank] = dict(model=model, hist=hist, time=run_time, mem=run_mem, pct=pct,
                            trainable=trainable, total=total, raw_pct=raw_pct, resident_other_gb=foreign_gb)
    print(f"📌 [S8 r={rank}] trainable {trainable:,} = {pct:.4f}% (count_trainable: {raw_pct:.4f}%) | "
          f"time {run_time:.1f}s | peak mem {run_mem:.2f} GB (excl. {foreign_gb:.2f} GB other resident) | "
          f"best val {min(hist['val_loss']):.4f} vs LoRA {min(lora_runs[rank]['hist']['val_loss']):.4f} | "
          f"mem vs LoRA {run_mem - lora_runs[rank]['mem']:+.2f} GB "
          f"({run_mem / lora_runs[rank]['mem'] - 1 if lora_runs[rank]['mem'] > 0 else float('nan'):+.0%})")

    print("\nEpoch |  Train Loss | Val Loss")
    print("-" * 34)
    for ep, (tl, vl) in enumerate(zip(hist["train_loss"], hist["val_loss"]), 1):
        print(f"  {ep:2d}  |   {tl:.4f}   | {vl:.4f}")
    print()

    free_memory()   # clear before the next rank

### ✍️ Step 8 Reflection  <font color="red">**[6 marks]**</font>

**[Your Answer]** must cover:
- Each `BitsAndBytesConfig` setting in plain language — `load_in_4bit`, `nf4`, double quantisation, and `compute_dtype=float16` — and why compute is fp16 on a T4. <font color="red">**[2 marks]**</font>
- What `prepare_model_for_kbit_training` does and why it is required before attaching LoRA. <font color="red">**[1 marks]**</font>
- QLoRA vs LoRA at the **same rank**: how much peak memory did 4-bit save, and did it cost any measurable answer quality? Cite both. <font color="red">**[2 marks]**</font>
- Why QLoRA's trainable-parameter count matches LoRA's at the same rank (only the frozen backbone's precision changed). <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

### ✅ Your Answer — Step 8

**`BitsAndBytesConfig`, setting by setting.**
- `load_in_4bit=True`: the frozen weight of every decoder `nn.Linear` is stored in 4 bits instead of 32. Those ≈357.8M weights drop from about 1.43 GB to about 0.18 GB. The embedding (tied LM head) is not quantised.
- `bnb_4bit_quant_type="nf4"`: NormalFloat-4 places its 16 code values at quantiles of a standard normal distribution. Each block of 64 weights is scaled by its absmax and mapped to the nearest code. Pre-trained weights are roughly zero-mean Gaussian, so NF4 puts its levels where weights are dense and loses less information than uniform INT4 or FP4.
- `bnb_4bit_use_double_quant=True`: every 64-weight block has its own fp32 absmax scale, which costs 0.5 bit per weight. Double quantisation stores those scales in 8 bits (in groups of 256), cutting the overhead to about 0.127 bit per weight. That saves about 0.37 bit per parameter.
- `bnb_4bit_compute_dtype=torch.float16`: 4-bit is a storage format only. At each matmul a block is de-quantised to fp16 and multiplied in fp16. **fp16 is used because the T4 (Turing, sm_75) has fp16 tensor cores but no bf16 support** (bf16 needs Ampere, sm_80+). bf16 would error or be slowly emulated, and fp32 compute would be several times slower. With fp32 LoRA weights and fp32 norms, fp16 compute is stable.

**What `prepare_model_for_kbit_training` does, and why it must come first.**
1. It freezes every base parameter (`requires_grad = False`).
2. It up-casts the remaining non-quantised fp16 tensors (RMSNorm weights, embeddings/LM head, q/k/v biases) to fp32, so normalisation and the logits are computed stably.
3. It enables gradient checkpointing. For the reentrant variant it also makes the embedding output require grad, so gradients can flow back through frozen, checkpointed blocks to the adapters.

It must run **before** `get_peft_model` because step 1 freezes *every* parameter that exists at that moment. Called afterwards, it would freeze (and cast) the LoRA A/B matrices too, leaving nothing to train.

**QLoRA vs LoRA at the same rank.** Peak memory is isolated: other fp32 models were parked on CPU and any other resident GPU memory was subtracted.

| | LoRA | QLoRA | Change |
|---|---|---|---|
| r = 8, peak memory | ⟦⟧ GB | ⟦⟧ GB | **⟦S12 r=8 mem %⟧** |
| r = 8, best val loss | ⟦⟧ | ⟦⟧ | |
| r = 8, ROUGE-L | ⟦⟧ | ⟦⟧ | |
| r = 8, BERTScore-F1 | ⟦⟧ | ⟦⟧ | |
| r = 16, peak memory | ⟦⟧ GB | ⟦⟧ GB | **⟦S12 r=16 mem %⟧** |
| r = 16, best val loss | ⟦⟧ | ⟦⟧ | |
| r = 16, ROUGE-L | ⟦⟧ | ⟦⟧ | |
| r = 16, BERTScore-F1 | ⟦⟧ | ⟦⟧ | |

QLoRA took **⟦S12 time Δ⟧** s longer per run because of on-the-fly de-quantisation.

**Quality cost:** **⟦pick one: no measurable cost, since the differences are within ±0.01 / a small cost of X ROUGE-L⟧**.

The memory saving is much smaller than "8×", for two reasons:
- Only the decoder linear weights are 4-bit. The 136M-parameter embedding is 27.6% of this small model, and `prepare_model_for_kbit_training` keeps it in fp32, at 0.54 GB.
- At batch 4 × 256, the 151,936-way logits and log-softmax (≈0.62 GB per fp32 copy) and the activations are a large share of the peak, and they do not depend on the backbone's precision.

QLoRA's advantage grows with model size, where weights dominate.

**Why the trainable count matches LoRA.** QLoRA uses the identical `LoraConfig`: same r, α = 2r, dropout 0.05 and the same 7 target modules, on the same architecture. The adapter matrices therefore have identical shapes: **4,399,104** trainable at r = 8 and **8,798,208** at r = 16. The notebook asserts both the identical config and the identical count. Only the storage precision of the *frozen* W changed.

`print_trainable_parameters()` reports the same 0.8826% and 1.7497% as LoRA. The provided `count_trainable()` reports **⟦S8 count_trainable % r=8⟧** and **⟦r=16⟧** instead, because a `Params4bit` tensor packs two 4-bit weights per uint8 element. `numel()` therefore counts the quantised layers at half their size: the *denominator* shrinks, not the trainable count. Absolute trainable counts are the meaningful comparison.

## <font color="blue"><b>Step 9: Test Evaluation — IFT, LoRA, QLoRA</b></font>  <font color="red">**[10 marks]**</font>

Now measure answer quality on the held-out test set. Greedy-generate an answer for a fixed slice of test prompts (`EVAL_N`, the same prompts for every model so scores are comparable), then score the generations against the references with two complementary metrics:

- **ROUGE-L** — overlap of word sequences (did we *say* similar things?).
- **BERTScore-F1** — embedding similarity (did we *mean* similar things, even with different wording?).

Score every run — full IFT, both LoRA ranks, and both QLoRA ranks — and collect the numbers (plus each run's % trainable, peak memory, and training time) into `results`, with the generated text kept in `preds_store` for the error analysis.

**Marks:** code <font color="red">**[6 marks]**</font> — correct `generate_answers` (greedy, prompt-only decode), ROUGE-L + BERTScore scoring, and scoring all five runs into `results` / `preds_store`; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 9: Test Evaluation — IFT, LoRA, QLoRA
# ============================================================
import evaluate as hf_evaluate
rouge = hf_evaluate.load("rouge")

GEN_BATCH = 8    # prompts decoded together (left-padded); identical for every model

@torch.no_grad()
def generate_answers(model, frame, max_new_tokens=200, n=EVAL_N):
    """Greedy-decode an answer for each prompt. Returns (predictions, references)."""

    n = min(n, len(frame))                                    # never more rows than the test split holds
    rows = frame.iloc[:n]                                     # the same fixed slice for every model
    prompts = [build_prompt(r["instruction"], r["input"]) for _, r in rows.iterrows()]
    refs = [r["output"].strip() for _, r in rows.iterrows()]

    parked = next(model.parameters()).device.type != torch.device(DEVICE).type
    if parked:
        model.to(DEVICE)                                      # parked fp32 models visit the GPU only to decode
    model.eval()
    stop_ids = list(dict.fromkeys([tokenizer.eos_token_id, tokenizer.pad_token_id]))   # <|im_end|>, <|endoftext|>
    preds = []
    tokenizer.padding_side = "left"                           # every prompt must end where generation starts
    try:
        for i in range(0, n, GEN_BATCH):
            enc = tokenizer(prompts[i:i + GEN_BATCH], return_tensors="pt", padding=True,
                            add_special_tokens=False).to(DEVICE)
            out = model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                                 max_new_tokens=max_new_tokens,
                                 do_sample=False, num_beams=1,              # greedy decoding
                                 temperature=None, top_p=None, top_k=None,  # neutralise Qwen's sampling defaults
                                 repetition_penalty=1.0,
                                 eos_token_id=stop_ids, pad_token_id=tokenizer.pad_token_id, use_cache=True)
            new_tokens = out[:, enc["input_ids"].shape[1]:]   # prompt-only decode: drop the echoed prompt
            preds += [t.strip() for t in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]
    finally:
        tokenizer.padding_side = "right"                      # collate() relies on right padding
    if parked:
        model.to("cpu")
        free_memory()

    return preds, refs


def score(preds, refs):
    """ROUGE-L (word overlap) + BERTScore-F1 (meaning similarity)."""

    rl = rouge.compute(predictions=preds, references=refs, rouge_types=["rougeL"], use_stemmer=True)["rougeL"]
    if not hasattr(score, "bert"):                            # load roberta-large once, reuse for all five runs
        from bert_score import BERTScorer
        score.bert = BERTScorer(lang="en", device=DEVICE)     # default English model: roberta-large, layer 17
    _, _, Fb = score.bert.score(preds, refs, batch_size=16)

    return {"rougeL": round(float(rl), 4), "bertscore_f1": round(float(Fb.mean()), 4)}


results, preds_store = {}, {}   # name -> metrics ; name -> predictions

In [ ]:
# ---- Full IFT ----
full_preds, test_refs = generate_answers(full_model, test_df)
results["full_ift"] = {**score(full_preds, test_refs),
                       "pct_trainable": round(pct_full, 2),
                       "peak_mem_gb": round(full_mem, 2),
                       "train_time_s": round(full_time, 1)}
preds_store["full_ift"] = full_preds
print("[full_ift]", results["full_ift"])

In [ ]:
# ---- LoRA and QLoRA, each at both ranks ----
for method, runs in [("lora", lora_runs), ("qlora", qlora_runs)]:
    for rank, run in runs.items():
        name = f"{method}_r{rank}"
        p, _ = generate_answers(run["model"], test_df)
        results[name] = {**score(p, test_refs),
                         "pct_trainable": round(run["pct"], 2),
                         "peak_mem_gb": round(run["mem"], 2),
                         "train_time_s": round(run["time"], 1)}
        preds_store[name] = p
        print(f"[{name}]", results[name])

### ✍️ Step 9 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- What ROUGE-L and BERTScore-F1 each measure, and why reporting both is more informative than either alone for free-form medical answers. <font color="red">**[2 marks]**</font>
- The scores across the five runs — which scores highest, and by how much over the others? <font color="red">**[1 marks]**</font>
- Since the references are machine-generated, why you should weigh *relative* differences between methods more than the *absolute* score values. <font color="red">**[1 marks]**</font>

*Cite the actual numbers your run produced.*

### ✅ Your Answer — Step 9

**What each metric measures.**
- **ROUGE-L** is the F-measure of the *longest common subsequence* of words between the generated answer and the reference (with stemming). It rewards using the same words in the same order, i.e. lexical and structural overlap. It is blind to synonyms and paraphrase, so a correct answer worded differently scores low.
- **BERTScore-F1** embeds both texts with RoBERTa-large (layer 17), matches each token to its most similar token in the other text by cosine similarity, and averages the matches into precision, recall and F1. It measures *semantic* similarity and tolerates paraphrase. However, its values fall in a narrow band (about 0.80–0.90 without baseline rescaling), and it can rate fluent-but-wrong text highly: a sentence with the wrong dose still "looks" similar.

Free-form medical answers have many valid phrasings, so each metric alone misleads:
- ROUGE-L alone undersells good paraphrases.
- BERTScore alone overlooks missing specifics.

Read together:
- High ROUGE and high BERTScore mean the same content in similar words.
- Low ROUGE with high BERTScore means the same meaning, different wording.
- Both low means different content.

Neither metric checks clinical correctness.

**Scores across the five runs.**

| Run | ROUGE-L | BERTScore-F1 |
|---|---|---|
| Full IFT | ⟦⟧ | ⟦⟧ |
| LoRA r = 8 | ⟦⟧ | ⟦⟧ |
| LoRA r = 16 | ⟦⟧ | ⟦⟧ |
| QLoRA r = 8 | ⟦⟧ | ⟦⟧ |
| QLoRA r = 16 | ⟦⟧ | ⟦⟧ |

The highest ROUGE-L is **⟦best run⟧**. It beats the runner-up **⟦2nd run⟧** by **⟦Δ ROUGE-L⟧** and the weakest run by **⟦Δ⟧**, and on BERTScore-F1 it **⟦also leads, by ⟦Δ⟧ / is not the leader⟧**. The spread across all five runs is only **⟦max − min ROUGE-L⟧** ROUGE-L and **⟦max − min BERTScore⟧** BERTScore. With 30 test prompts, gaps of about 0.01 are within noise.

**Why relative gaps matter more than absolute values.** All five runs use the same 30 prompts and the same greedy decoding, and they are scored against the same text-davinci-003 references. Those references have quirks: davinci's length and phrasing habits, and occasional errors. The quirks bias every run in the same direction, so they largely cancel when we compare methods.

The absolute numbers mostly measure resemblance to one machine's phrasing. A ROUGE-L of about 0.2–0.3 does not mean the answers are 70% wrong, and a clinically better answer can score lower than a fluent imitation of the reference. The informative signal is the *difference* between methods under the same yardstick.

## <font color="blue"><b>Step 10: Plots — Train vs Validation Loss</b></font>  <font color="red">**[6 marks]**</font>

Visualise how each run trained. Draw one panel per run — full IFT, both LoRA ranks, and both QLoRA ranks — showing per-epoch **train** and **validation** loss, all sharing the same y-axis so loss levels are directly comparable across methods and ranks. The curves come from each run's `hist` (`train_loss` / `val_loss`).

**Marks:** code <font color="red">**[4 marks]**</font> — one train-vs-val panel per run, shared y-axis, correct labels/legend/titles; reflection <font color="red">**[2 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 10: Plots — Train vs Validation Loss (per epoch)
# ============================================================

run_hists = ([("Full IFT", full_hist)]
             + [(f"LoRA r={r}", lora_runs[r]["hist"]) for r in LORA_RANKS]
             + [(f"QLoRA r={r}", qlora_runs[r]["hist"]) for r in LORA_RANKS])
C_TRAIN, C_VAL, INK = "#2a78d6", "#eb6834", "#52514e"

fig, axes = plt.subplots(1, len(run_hists), figsize=(4.0 * len(run_hists), 4.2), sharey=True)
for ax, (name, h) in zip(axes, run_hists):
    ep = np.arange(1, len(h["train_loss"]) + 1)
    ax.plot(ep, h["train_loss"], "-o", color=C_TRAIN, lw=2, ms=8, label="train")
    ax.plot(ep, h["val_loss"], "--s", color=C_VAL, lw=2, ms=8, label="validation")
    b = int(np.argmin(h["val_loss"]))
    ax.annotate(f"best val {h['val_loss'][b]:.3f}", (ep[b], h["val_loss"][b]), textcoords="offset points",
                xytext=(0, -16), ha="center", fontsize=9, color=INK)
    ax.set_title(name)
    ax.set_xlabel("Epoch")
    ax.set_xticks(ep)
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=9)
axes[0].set_ylabel("Answer-token NLL (lower is better)")
fig.suptitle("Train vs validation loss per epoch — shared y-axis across all five runs", y=1.02)
plt.tight_layout()
plt.show()

# Validation curves overlaid: which run converges fastest / lowest?
style = {"Full IFT": ("#2a78d6", "-"), "LoRA r=8": ("#eb6834", "--"), "LoRA r=16": ("#eb6834", "-"),
         "QLoRA r=8": ("#1baf7a", "--"), "QLoRA r=16": ("#1baf7a", "-")}
fig, ax = plt.subplots(figsize=(8, 4.2))
for name, h in run_hists:
    col, ls = style[name]
    ax.plot(np.arange(1, len(h["val_loss"]) + 1), h["val_loss"], ls, marker="o", color=col, lw=2, ms=8, label=name)
ax.set_title("Validation loss per epoch, all runs")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation NLL")
ax.set_xticks(np.arange(1, EPOCHS + 1))
ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=2)
plt.tight_layout(); plt.show()

loss_tbl = pd.DataFrame({name: {"train_e1": h["train_loss"][0], "train_last": h["train_loss"][-1],
                                "val_e1": h["val_loss"][0], "val_last": h["val_loss"][-1],
                                "val_best": min(h["val_loss"]), "best_epoch": int(np.argmin(h["val_loss"])) + 1,
                                "val_minus_train_last": h["val_loss"][-1] - h["train_loss"][-1]}
                         for name, h in run_hists}).T
loss_tbl["best_epoch"] = loss_tbl["best_epoch"].astype(int)
print("📌 [S10] loss summary per run")
print(loss_tbl.round(4).to_string())

### ✍️ Step 10 Reflection  <font color="red">**[2 marks]**</font>

**[Your Answer]** must cover:
- Which method converged fastest and to the lowest validation loss, and whether any run shows signs of **overfitting** (validation rising while training keeps falling). <font color="red">**[1 marks]**</font>
- Whether a lower *training* loss reliably predicted better *test* answers back in Step 9 — and what that tells you. <font color="red">**[1 marks]**</font>

*Cite the curves and the Step 9 numbers.*

### ✅ Your Answer — Step 10

**Convergence and overfitting.** The overlaid validation curves and the `📌 [S10]` table show that:
- **⟦run⟧** reaches the lowest validation loss (**⟦val_best⟧** at epoch **⟦⟧**).
- **⟦run⟧** converges fastest, with the biggest drop by epoch 1 or 2 (val_e1 = **⟦⟧**).

The adapter runs use the 20× larger LR, so their curves fall faster per epoch than full IFT's (1e-5). Full IFT moves from **⟦val_e1⟧** to **⟦val_last⟧**.

Overfitting, meaning validation rising while training keeps falling, **⟦pick one: does not appear in any run, since every validation curve is still flat or falling at epoch 3 and the val − train gaps stay small (⟦range⟧) / appears in ⟦run⟧, where validation rises from ⟦⟧ to ⟦⟧ while train falls from ⟦⟧ to ⟦⟧, and early stopping restored epoch ⟦⟧⟧**.

**Did lower training loss predict better test answers?** The lowest final *training* loss belongs to **⟦run, train_last⟧**, but the best test ROUGE-L belongs to **⟦run, ROUGE-L⟧**. Ordered by training loss, the runs **⟦pick one: do / do not⟧** follow the Step 9 ranking.

The lesson is that training loss measures how well a model fits 300 seen examples under teacher forcing. Test quality depends on *generalisation* and on *free-running greedy generation*, including when the model stops and whether it rambles. That is why the held-out validation loss and the generation metrics have to be checked, not the training curve. Even validation NLL is only a proxy: it scores each next token given the *reference* prefix, which the model never sees at test time.

## <font color="blue"><b>Step 11: Error Analysis</b></font>  <font color="red">**[8 marks]**</font>

Numbers alone don't tell you *how* a model is wrong. Take the best run by **ROUGE-L**, print a few of its generated answers next to the references, and back that up with a quantitative summary of the failure modes: predicted-vs-reference length, the length ratio (a quick proxy for rambling or truncation), and counts of empty / much-shorter / much-longer generations.

**Marks:** code <font color="red">**[4 marks]**</font> — select the best run by ROUGE-L, show sample generations vs references, and compute the length-ratio + failure-mode summary; reflection <font color="red">**[4 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 11: Error Analysis
# ============================================================
import textwrap

best_name = max(results, key=lambda k: results[k]["rougeL"])
best_preds = preds_store[best_name]
print(f"Best run by ROUGE-L: {best_name} -> {results[best_name]}\n")

per_ex_rl = np.array(rouge.compute(predictions=best_preds, references=test_refs, rouge_types=["rougeL"],
                                   use_stemmer=True, use_aggregator=False)["rougeL"], dtype=float)
order = np.argsort(per_ex_rl)
samples = [("highest ROUGE-L", order[-1]), ("upper-middle", order[(3 * len(order)) // 4]),
           ("median", order[len(order) // 2]), ("lowest ROUGE-L", order[0])]

def _wrap(s, limit=900):
    s = s if len(s) <= limit else s[:limit] + " [...]"
    return "\n".join(textwrap.fill(line, 110) for line in s.splitlines() if line.strip())

for label, i in samples:
    r = test_df.iloc[i]
    print("=" * 112)
    print(f"[{label}] test #{i} | ROUGE-L {per_ex_rl[i]:.3f} | words: model {len(best_preds[i].split())} "
          f"vs reference {len(test_refs[i].split())}")
    print("-" * 112)
    print("INSTRUCTION:", _wrap(r["instruction"], 400))
    if r["input"]:
        print("INPUT:", _wrap(r["input"], 400))
    print("\nREFERENCE (text-davinci-003):\n" + _wrap(test_refs[i]))
    print(f"\nMODEL ANSWER ({best_name}):\n" + (_wrap(best_preds[i]) or "<EMPTY>"))
print("=" * 112)

In [ ]:
# ---- Quantitative summary of the errors ----
import re
CAVEAT_RE = (r"consult|healthcare (?:provider|professional)|medical (?:advice|attention|professional)|"
             r"see a (?:doctor|physician)|seek (?:immediate |urgent )?(?:medical )?(?:help|care|attention)")
GEN_CAP = 200   # max_new_tokens used in generate_answers

def error_profile(preds, refs):
    pw = np.array([len(p.split()) for p in preds])
    rw = np.array([len(r.split()) for r in refs])
    ptok = np.array([len(tokenizer(p, add_special_tokens=False)["input_ids"]) for p in preds])
    ratio = pw / np.maximum(rw, 1)
    return pd.DataFrame({"pred_words": pw, "ref_words": rw, "len_ratio": ratio,
                         "hit_token_cap": ptok >= GEN_CAP - 2,
                         "pred_caveat": [bool(re.search(CAVEAT_RE, p, re.I)) for p in preds],
                         "ref_caveat":  [bool(re.search(CAVEAT_RE, r, re.I)) for r in refs]})

prof = error_profile(best_preds, test_refs)
prof["rougeL"] = per_ex_rl
n = len(prof)
summary = {
    "examples": n,
    "mean pred words": prof["pred_words"].mean(), "mean ref words": prof["ref_words"].mean(),
    "median length ratio (pred/ref)": prof["len_ratio"].median(), "mean length ratio": prof["len_ratio"].mean(),
    "empty generations": int((prof["pred_words"] == 0).sum()),
    "much shorter (<0.5x ref)": int((prof["len_ratio"] < 0.5).sum()),
    "much longer (>2x ref)": int((prof["len_ratio"] > 2.0).sum()),
    f"hit {GEN_CAP}-token cap (cut off)": int(prof["hit_token_cap"].sum()),
    "answers with a safety caveat": int(prof["pred_caveat"].sum()),
    "references with a safety caveat": int(prof["ref_caveat"].sum()),
    "corr(len_ratio, ROUGE-L)": prof["len_ratio"].corr(prof["rougeL"]),
}
print(f"Failure-mode summary for {best_name} on {n} test prompts")
for k, v in summary.items():
    print(f"  {k:<36} {v:.2f}" if isinstance(v, float) else f"  {k:<36} {v}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(prof["ref_words"], prof["pred_words"], s=64, color="#2a78d6", edgecolor="white", linewidth=2)
lim = max(prof["ref_words"].max(), prof["pred_words"].max()) * 1.05
axes[0].plot([0, lim], [0, lim], ":", color="#52514e", lw=1.5, label="pred = ref")
axes[0].set(xlabel="Reference length (words)", ylabel="Model answer length (words)",
            title=f"{best_name}: answer vs reference length")
axes[0].legend(frameon=False)
axes[1].hist(prof["len_ratio"], bins=15, color="#2a78d6", edgecolor="white", linewidth=1.5)
for x in (0.5, 2.0):
    axes[1].axvline(x, color="#eb6834", ls="--", lw=2)
axes[1].set(xlabel="Length ratio (pred / ref words)", ylabel="Examples",
            title="Length ratio (dashed: 0.5x and 2x thresholds)")
for ax in axes:
    ax.grid(alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Same profile for every run, so the best run's behaviour can be put in context.
rows = {}
for name, p in preds_store.items():
    pr = error_profile(p, test_refs)
    rows[name] = {"rougeL": results[name]["rougeL"], "mean_pred_words": pr["pred_words"].mean(),
                  "median_len_ratio": pr["len_ratio"].median(), "empty": int((pr["pred_words"] == 0).sum()),
                  "much_shorter": int((pr["len_ratio"] < 0.5).sum()), "much_longer": int((pr["len_ratio"] > 2).sum()),
                  "hit_cap": int(pr["hit_token_cap"].sum()), "caveats": int(pr["pred_caveat"].sum())}
print("\n📌 [S11] length / failure profile, all runs (mean reference length "
      f"{prof['ref_words'].mean():.0f} words)")
print(pd.DataFrame(rows).T.round(3).to_string())

### ✍️ Step 11 Reflection  <font color="red">**[4 marks]**</font>

**[Your Answer]** must cover:
- One recurring way the model's answers differ from the references — length, specificity, hedging, or missing safety caveats — with a concrete example from the samples. <font color="red">**[2 marks]**</font>
- What the length-ratio and failure counts reveal about the model's behaviour (rambling, truncation, empty generations). <font color="red">**[1 marks]**</font>
- One concrete change you would try to reduce these errors (e.g. more epochs, higher rank, longer `MAX_LEN`, or different decoding settings). <font color="red">**[1 marks]**</font>

*Cite the samples and the summary numbers your run produced.*

### ✅ Your Answer — Step 11

**A recurring difference from the references.** The best run by ROUGE-L is **⟦best run⟧** (ROUGE-L **⟦⟧**). Its most consistent deviation is **⟦pick the one your samples show, e.g. length: its answers are shorter and less exhaustive than davinci's / specificity: it gives generic advice where the reference names drugs, doses or tests / hedging or safety: it adds or omits "consult a healthcare professional"⟧**.

A concrete example: on test #⟦⟧ (*"⟦instruction⟧"*), the reference **⟦quote 1–2 lines⟧**, while the model **⟦quote 1–2 lines⟧**. That example has a ROUGE-L of **⟦⟧** and a length ratio of **⟦⟧**. The safety-caveat counts point the same way: **⟦⟧ of 30** model answers contain a caveat, versus **⟦⟧ of 30** references.

**What the length statistics reveal.**

| | Value |
|---|---|
| Mean model answer length | **⟦⟧ words** |
| Mean reference length | **⟦⟧ words** |
| Median length ratio (model / reference) | **⟦⟧** |
| Empty generations | **⟦⟧** |
| Much shorter (< 0.5×) | **⟦⟧** |
| Much longer (> 2×) | **⟦⟧** |
| Hit the 200-token cap | **⟦⟧** |
| corr(length ratio, ROUGE-L) | **⟦⟧** |

So the dominant failure is **⟦pick one: truncation or terseness, i.e. answers stop well short of the reference and the cap clips ⟦⟧ of them / rambling, i.e. answers overshoot the reference length⟧**, while empty generations are **⟦none / rare⟧**. The model has learned to stop (EOS was supervised), and ROUGE-L losses come mainly from **⟦length mismatch / wording differences at similar length⟧**.

**One concrete change.** **⟦pick the one that matches your numbers⟧**:
- **If answers are clipped or too short:** raise `max_new_tokens` from 200 to about 320, and `MAX_LEN` to 384 for the adapter runs, so long references are not truncated during training. Truncation currently teaches the model to emit EOS mid-answer on long examples.
- **If answers ramble or repeat:** keep greedy decoding but add `repetition_penalty=1.1` / `no_repeat_ngram_size=4`, and train one or two more epochs at LoRA r = 16, which still had falling validation loss.

## <font color="blue"><b>Step 12: Parameter Analysis</b></font>  <font color="red">**[6 marks]**</font>

Bring everything together into the cost–quality picture. Build one comparison table across all five runs — **quality** (ROUGE-L, BERTScore-F1) against **cost** (% trainable parameters, peak GPU memory, training time) — then draw the supporting charts: quality bars with peak memory annotated, and a trainable-% bar chart that makes the full-IFT-vs-adapter gap obvious.

**Marks:** code <font color="red">**[4 marks]**</font> — the comparison table over all runs and the quality + trainable-% charts; reflection <font color="red">**[2 marks]**</font> — see below.

In [ ]:
# ============================================================
#  Step 12: Parameter Analysis
# ============================================================

# ---- Comparison table: quality vs cost across every run ----
RUN_ORDER = ["full_ift"] + [f"lora_r{r}" for r in LORA_RANKS] + [f"qlora_r{r}" for r in LORA_RANKS]
LABELS = {"full_ift": "Full IFT", **{f"lora_r{r}": f"LoRA r={r}" for r in LORA_RANKS},
          **{f"qlora_r{r}": f"QLoRA r={r}" for r in LORA_RANKS}}
abs_trainable = {"full_ift": tr_full,
                 **{f"lora_r{r}": lora_runs[r]["trainable"] for r in LORA_RANKS},
                 **{f"qlora_r{r}": qlora_runs[r]["trainable"] for r in LORA_RANKS}}
best_val = {"full_ift": min(full_hist["val_loss"]),
            **{f"lora_r{r}": min(lora_runs[r]["hist"]["val_loss"]) for r in LORA_RANKS},
            **{f"qlora_r{r}": min(qlora_runs[r]["hist"]["val_loss"]) for r in LORA_RANKS}}

cmp_df = pd.DataFrame(results).T.loc[RUN_ORDER, ["rougeL", "bertscore_f1", "pct_trainable", "peak_mem_gb", "train_time_s"]]
cmp_df = cmp_df.astype(float)
cmp_df.insert(2, "trainable_params", pd.Series(abs_trainable).astype(int))
cmp_df.insert(2, "best_val_loss", pd.Series(best_val).round(4))
ref = cmp_df.loc["full_ift"]
cmp_df["rougeL_vs_full"] = (cmp_df["rougeL"] - ref["rougeL"]).round(4)
cmp_df["mem_vs_full"] = (cmp_df["peak_mem_gb"] / ref["peak_mem_gb"]).round(2)
cmp_df["time_vs_full"] = (cmp_df["train_time_s"] / ref["train_time_s"]).round(2)
cmp_df = cmp_df.sort_values("rougeL", ascending=False)
cmp_df.index = [LABELS[i] for i in cmp_df.index]
print("Quality vs cost (sorted by ROUGE-L; peak memory is the isolated per-run CUDA peak)")
display(cmp_df.style.format({"rougeL": "{:.4f}", "bertscore_f1": "{:.4f}", "best_val_loss": "{:.4f}",
                             "trainable_params": "{:,}", "pct_trainable": "{:.2f}%", "peak_mem_gb": "{:.2f}",
                             "train_time_s": "{:.1f}", "rougeL_vs_full": "{:+.4f}", "mem_vs_full": "{:.2f}x",
                             "time_vs_full": "{:.2f}x"}))

for r in LORA_RANKS:
    l, q = results[f"lora_r{r}"], results[f"qlora_r{r}"]
    print(f"📌 [S12 r={r}] QLoRA vs LoRA: mem {q['peak_mem_gb'] - l['peak_mem_gb']:+.2f} GB "
          f"({q['peak_mem_gb'] / l['peak_mem_gb'] - 1 if l['peak_mem_gb'] > 0 else float('nan'):+.0%}) | "
          f"time {q['train_time_s'] - l['train_time_s']:+.1f}s | "
          f"ROUGE-L {q['rougeL'] - l['rougeL']:+.4f} | BERTScore {q['bertscore_f1'] - l['bertscore_f1']:+.4f}")
for m in ["lora", "qlora"]:
    a, b = results[f"{m}_r8"], results[f"{m}_r16"]
    print(f"📌 [S12 {m}] r16 vs r8: ROUGE-L {b['rougeL'] - a['rougeL']:+.4f} | BERTScore {b['bertscore_f1'] - a['bertscore_f1']:+.4f} | "
          f"mem {b['peak_mem_gb'] - a['peak_mem_gb']:+.2f} GB | time {b['train_time_s'] - a['train_time_s']:+.1f}s")

In [ ]:
# ---- Quality chart with peak memory annotated ----
METHOD_COLOR = {"full_ift": "#2a78d6", "lora_r8": "#eb6834", "lora_r16": "#eb6834",
                "qlora_r8": "#1baf7a", "qlora_r16": "#1baf7a"}
x = np.arange(len(RUN_ORDER))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, metric, title in zip(axes, ["rougeL", "bertscore_f1"], ["ROUGE-L (word overlap)", "BERTScore-F1 (meaning)"]):
    vals = [results[n][metric] for n in RUN_ORDER]
    bars = ax.bar(x, vals, width=0.6, color=[METHOD_COLOR[n] for n in RUN_ORDER], edgecolor="white", linewidth=2)
    for b, n in zip(bars, RUN_ORDER):
        ax.annotate(f"{results[n][metric]:.3f}\n{results[n]['peak_mem_gb']:.2f} GB peak",
                    (b.get_x() + b.get_width() / 2, b.get_height()), xytext=(0, 4), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9, color="#0b0b0b")
    ax.set_xticks(x, [LABELS[n] for n in RUN_ORDER])
    ax.set_ylim(0, max(max(vals), 1e-3) * 1.25)
    ax.set_title(title)
    ax.set_ylabel(metric)
    ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#2a78d6", "#eb6834", "#1baf7a"]]
axes[1].legend(handles, ["Full IFT (fp32)", "LoRA (fp32 backbone)", "QLoRA (4-bit NF4 backbone)"],
               frameon=False, loc="lower right")
fig.suptitle("Test-set answer quality per run (label: score / isolated peak GPU memory)", y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ---- Trainable-parameters comparison (the LoRA/QLoRA story) ----
fig, ax = plt.subplots(figsize=(9, 4.8))
pcts = [results[n]["pct_trainable"] for n in RUN_ORDER]
bars = ax.bar(x, pcts, width=0.6, color=[METHOD_COLOR[n] for n in RUN_ORDER], edgecolor="white", linewidth=2)
for b, n in zip(bars, RUN_ORDER):
    ax.annotate(f"{results[n]['pct_trainable']:.2f}%\n{abs_trainable[n] / 1e6:.1f}M params",
                (b.get_x() + b.get_width() / 2, b.get_height()), xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=9, color="#0b0b0b")
ax.set_yscale("log")
ax.set_ylim(0.1, 600)
ax.set_xticks(x, [LABELS[n] for n in RUN_ORDER])
ax.set_ylabel("% of parameters trained (log scale)")
ax.set_title(f"Trainable parameters: full IFT updates {tr_full / 1e6:.0f}M, adapters "
             f"{abs_trainable['lora_r8'] / 1e6:.1f}M (r=8) / {abs_trainable['lora_r16'] / 1e6:.1f}M (r=16)")
ax.grid(axis="y", alpha=0.3, which="major"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### ✍️ Step 12 Reflection  <font color="red">**[2 marks]**</font>

**[Your Answer]** must cover:
- Which method gives the best **balance** of quality, trainable-%, memory, and training time for a free T4 — not simply the highest score — citing the comparison table. <font color="red">**[1 mark]**</font>
- How **rank** (8 vs 16) and **quantisation** (LoRA vs QLoRA) each traded answer quality against cost. <font color="red">**[1 mark]**</font>

*Cite the actual numbers from your comparison table.*

### ✅ Your Answer — Step 12

**Best balance on a free T4.** From the comparison table, **⟦pick one, e.g. QLoRA r = 8 / LoRA r = 8⟧** gives the best balance. It reaches ROUGE-L **⟦⟧** and BERTScore-F1 **⟦⟧**, which is within **⟦Δ⟧** ROUGE-L of the top run (**⟦top run⟧**, **⟦⟧**). It does so while:
- training only **4.4M parameters (0.88%)**, versus 494M (100%) for full IFT;
- peaking at **⟦⟧ GB**, which is **⟦mem_vs_full⟧×** full IFT's ⟦⟧ GB;
- training in **⟦⟧ s**, which is **⟦time_vs_full⟧×** full IFT's ⟦⟧ s.

Full IFT costs about **⟦⟧ GB** and leaves little head-room on a 15 GB T4. Its fp32 weights, gradients and AdamW states alone take about 7.9 GB, so it would not scale to a larger model, and it buys **⟦+/− Δ⟧** ROUGE-L. An adapter run, by contrast, also produces a checkpoint of a few MB instead of 2 GB. **⟦If QLoRA was picked: the extra ⟦⟧ GB saved over LoRA is what lets the same recipe scale to a bigger backbone on the same T4, at a cost of ⟦⟧ s more training time. If LoRA was picked: at 0.5B the 4-bit saving (⟦⟧ GB) is small, and LoRA is faster with ⟦equal / slightly better⟧ quality.⟧**

**How rank and quantisation traded quality against cost.**
- **Rank, 8 → 16:** doubles the adapter parameters (4.4M → 8.8M, 0.88% → 1.75%). It costs **⟦S12 lora mem Δ⟧ GB** of memory and **⟦time Δ⟧ s** of time. The quality change is **⟦S12 lora r16 vs r8: ROUGE-L Δ / BERTScore Δ⟧** for LoRA and **⟦S12 qlora …⟧** for QLoRA. That is **⟦a marginal gain / no reliable gain⟧**, so the extra capacity is **⟦worth it / not needed⟧** for 300 examples.
- **Quantisation, LoRA → QLoRA:** at the same rank it cuts peak memory by **⟦S12 r=8 mem %⟧** (r = 8) and **⟦S12 r=16 mem %⟧** (r = 16). It costs **⟦S12 time Δ⟧ s** more training time because of de-quantisation. ROUGE-L changes by **⟦S12 r=8 / r=16 ROUGE Δ⟧**, which is **⟦no measurable / a small⟧** quality cost.

The trainable-parameter count is unchanged by quantisation, so on this task quantisation trades time for memory, and rank trades parameters for **⟦little / some⟧** quality.

In [ ]:
# END OF THE NOTEBOOK --------------------------------